# FFLogs Guilds Pipeline Draft (How-to-use)

1. Select the transformation stage using the widget `stage`.
2. Run the notebook to review the pipeline output for the selected stage at the bottom cell.

In [0]:
import json

from pyspark.sql import (
    DataFrame,
    Window,
    Column
)
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
%reload_ext autoreload

In [0]:
import os, sys
sys.path.append("/Workspace/Users/stan.mng@gmail.com/fflogs_graphql/jobs/")

from settings import fflogs_settings as settings
from schemas import (
    VolumeContent,
    BronzeTable,
    SilverTable,
    GoldTable
)
from utils import (
    to_title_case,
    trim_whitespace,
    roundfloat
)

In [0]:
def define_pipeline_widget():
    dbutils.widgets.dropdown(
        name="stage",
        defaultValue="Gold",
        choices=["Raw", "Bronze", "Silver", "Gold"],
        label="Select transformation stage"
    )


In [0]:
class FFLogsGuildsVolumeRaw(VolumeContent):
    def __init__(self, volume_dir: str, source: str) -> None:
        super().__init__(volume_dir, source)

    def read(self) -> "FFLogsGuildsVolumeRaw":
        file = next(
            f for f in os.listdir(self.volume_dir)
            if f.endswith(".json")
        )
        with open(os.path.join(self.volume_dir, file), "r") as f:
            self.content = json.load(f)

        return self
    
    def preview(self) -> dict | str | None:
        preview_data = self.content["guildData"]["guilds"]["data"]
        return {
            "guildData": {
                "guilds": {
                    "data": preview_data[:3]
                }
            }
        }

In [0]:
class FFLogsGuildsBronze(BronzeTable):

    def __init__(self, raw: VolumeContent) -> None:
        super().__init__(raw)

    def ingest(self) -> "FFLogsGuildsBronze":
        self.df = (spark.read
            .format("json")
            .option("multiLine", True)
            .load(f"{self.raw.volume_dir}/*.json")
        )
        return self

    def attach_metadata(self) -> "FFLogsGuildsBronze":
        self.df = (self.df
            .withColumn("ldts", F.current_timestamp())
            .withColumn("rsrc", F.lit(self.raw.source))
        )
        return self

root
 |-- guildData: struct (nullable = true)
 |    |-- guilds: struct (nullable = true)
 |    |    |-- data: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- description: string (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |    |-- server: struct (nullable = true)
 |    |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |    |-- region: struct (nullable = true)
 |    |    |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- type: string (nullable = true)
 |    |    |    |    |-- zoneRanking: struct (nullable = true)
 |    |    |    |    |    |-- progress: struct (nullable = true)
 |    |    |    |    |    |    |-- regionRank: struct (nullable = true)
 |    |    |    |    |    |    |    |-- number: long (nullable = true)
 |    |    |    |    |    |    |-- worldRank: struct (nullable = true)
 |    |    |    |    |    |    |    |-- number: long (nullable = true)
 |    |    |    |    |    |-- speed: struct (nullable = true)
 |    |    |    |    |    |    |-- regionRank: struct (nullable = true)
 |    |    |    |    |    |    |    |-- number: long (nullable = true)
 |    |    |    |    |    |    |-- worldRank: struct (nullable = true)
 |    |    |    |    |    |    |    |-- number: long (nullable = true)
 |    |    |-- has_more_pages: boolean (nullable = true)
 |    |    |-- total: long (nullable = true)
 |-- ldts: timestamp (nullable = false)
 |-- rsrc: string (nullable = false)

In [0]:
class FFLogsGuildsSilver(SilverTable):
    def __init__(self, bronze: BronzeTable) -> None:
        super().__init__(bronze)

    def unpack(self) -> "FFLogsGuildsSilver":
        self.df = self.base_df.select(
            # ===== Metadata
            "ldts",
            "rsrc",
            # ===== Record Unpacking
            F.explode("guildData.guilds.data").alias("s")
        )
        return self

    def define(self) -> "FFLogsGuildsSilver":
        self.df = (self.df or self.base_df).select(
            # ===== Metadata
            "ldts",
            "rsrc",
            # ===== Record Definition
            F.col("s.id").alias("guild_id"),
            F.col("s.name").alias("guild_name"),
            F.col("s.description").alias("guild_description"),
            F.col("s.server.id").alias("server_id"),
            F.col("s.server.region.id").alias("region_id"),
            F.col("s.zoneRanking.progress.worldRank.number")
                .alias("world_prog_rank"),
            F.col("s.zoneRanking.progress.regionRank.number")
                .alias("region_prog_rank"),
            F.col("s.zoneRanking.speed.worldRank.number")
                .alias("world_speed_rank"),
            F.col("s.zoneRanking.speed.regionRank.number")
                .alias("region_speed_rank"),
        )
        return self

    def clean(self) -> "FFLogsGuildsSilver":
        self.df = self.df.select(
            # ===== Metadata
            "ldts",
            "rsrc",
            # ===== Guild
            F.col("guild_id").cast("int"),
            F.nullif(F.initcap(trim_whitespace(F.col("guild_name"))), F.lit("")).alias("guild_name"),
            F.nullif(trim_whitespace(F.col("guild_description")), F.lit("")).alias("guild_description"),
            # ===== Foreign Keys
            F.col("server_id").cast("int"),
            F.col("region_id").cast("int"),
            # ===== Rankings
            F.col("world_prog_rank").cast("int"),
            F.col("region_prog_rank").cast("int"),
            F.col("world_speed_rank").cast("int"),
            F.col("region_speed_rank").cast("int"),
        )
        return self

    def hash(self) -> "FFLogsGuildsSilver":
        return self

    def expect(self) -> "FFLogsGuildsSilver":
        self.df = self.df.filter(
            F.col("world_prog_rank").isNotNull() |
            F.col("region_prog_rank").isNotNull() |
            F.col("world_speed_rank").isNotNull() |
            F.col("region_speed_rank").isNotNull()
        )
        return self

    def deduplicate(self) -> "FFLogsGuildsSilver":
        window = (Window
            .partitionBy("guild_id")
            .orderBy(F.col("ldts").desc())
        )
        self.df = (self.df
            .withColumn("_rank", F.row_number().over(window))
            .filter(F.col("_rank") == 1)
            .drop("_rank")
        )
        return self

In [0]:
class FFLogsGuildsGold(GoldTable):
    def __init__(self, silver: SilverTable) -> None:
        super().__init__(silver)

    def define(self) -> "FFLogsGuildsGold":
        self.df = (self.df or self.base_df).select(
            # ===== Guild
            F.col("guild_id"),
            F.col("guild_name"),
            F.col("guild_description"),
            # ===== Foreign Keys
            F.col("server_id"),
            F.col("region_id"),
            # ===== Rankings
            F.col("world_prog_rank"),
            F.col("region_prog_rank"),
            F.col("world_speed_rank"),
            F.col("region_speed_rank"),
        )
        return self

In [0]:
def draft_pipeline() -> tuple[
    VolumeContent, BronzeTable, SilverTable, GoldTable
]:
    raw = FFLogsGuildsVolumeRaw(
        volume_dir=settings.guilds_volume,
        source=settings.SECRET_SCOPE,
    ).read()

    bronze_table = (FFLogsGuildsBronze(raw)
        .ingest()
        .attach_metadata()
    )

    silver_table = (FFLogsGuildsSilver(bronze_table)
        .unpack()
        .define()
        .clean()
        .expect()
        .deduplicate()
    )

    gold_table = FFLogsGuildsGold(silver_table).define()

    return raw, bronze_table, silver_table, gold_table

In [0]:
def review_draft() -> None:
    stage = dbutils.widgets.get("stage").lower()
    raw, bronze, silver, gold = draft_pipeline()

    stages = {
        "raw": raw,
        "bronze": bronze,
        "silver": silver,
        "gold": gold,
    }

    target = stages[stage]
    match target:
        case _ if isinstance(target, VolumeContent):
            print(json.dumps(target.preview(), indent=4))
        case _ if isinstance(target, BronzeTable | SilverTable | GoldTable):
            base_df = getattr(target, "base_df", None)
            if base_df is not None:
                print(f"===== {stage} (base) =====")
                display(base_df)
                display(base_df.count())
                print("\n")

            if target.df is not None:
                print(f"===== {stage} =====")
                display(target.df)
                display(target.df.count())
                print("\n")

In [0]:
review_draft()